In [ ]:
import tensorflow as tf
import numpy as np


# import tensorflow as tf 
# from tensorflow.examples.tutorials.mnist 
# import input_data mnist = input_data.read_data_sets('MNIST_data', one_hot=True)
# 关闭 eager execution，兼容 TensorFlow 1.x 风格代码
tf.compat.v1.disable_eager_execution()

# 读取 MNIST 数据
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 数据预处理
x_train = x_train.reshape(-1, 784).astype('float32')
x_test = x_test.reshape(-1, 784).astype('float32')

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 2000

def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1.0})
    correct_prediction = tf.equal(tf.argmax(y_pre, 1), tf.argmax(v_ys, 1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1.0})
    return result

def weight_variable(shape):
    initial = tf.random.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    # 每一维度滑动步长全部是1，padding方式选择SAME
    return tf.nn.conv2d(x, W, strides=[1,1,1,1], padding='SAME')

def max_pool_2x2(x):
    # 滑动步长是2步；池化窗口高和宽都是2；padding方式选择SAME
    return tf.nn.max_pool(x, ksize=[1,2,2,1], strides=[1,2,2,1], padding='SAME')

# define placeholder for inputs to network
xs = tf.compat.v1.placeholder(tf.float32, [None, 784]) / 255.
ys = tf.compat.v1.placeholder(tf.float32, [None, 10])
keep_prob = tf.compat.v1.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1,28,28,1])

# 卷积层 1
## conv1 layer ##
W_conv1 = weight_variable([7,7,1,32]) # patch 7x7, in size 1, out size 32
b_conv1 = bias_variable([32])
h_conv1 = tf.nn.relu(conv2d(x_image,W_conv1)+b_conv1)
h_pool1 = max_pool_2x2(h_conv1)

# 卷积层 2
W_conv2 = weight_variable([5,5,32,64])  # patch 5x5, in size 32, out size 64
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2)+b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

# 全连接层 1
## fc1 layer ##
W_fc1 = weight_variable([7 * 7 * 64, 1024])
b_fc1 = bias_variable([1024])

h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)

# dropout：TF2 中建议使用 rate=1-keep_prob
h_fc1_drop = tf.nn.dropout(h_fc1, rate=1 - keep_prob)

# 全连接层 2
## fc2 layer ##
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])

logits = tf.matmul(h_fc1_drop, W_fc2) + b_fc2
prediction = tf.nn.softmax(logits)

# 交叉熵函数（更稳定的写法）
cross_entropy = tf.reduce_mean(
    tf.nn.softmax_cross_entropy_with_logits(labels=ys, logits=logits)
)

train_step = tf.compat.v1.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

with tf.compat.v1.Session() as sess:
    init = tf.compat.v1.global_variables_initializer()
    sess.run(init)

    batch_size = 100
    num_train = x_train.shape[0]

    for i in range(max_epoch):
        start = (i * batch_size) % num_train
        end = start + batch_size

        batch_xs = x_train[start:end]
        batch_ys = y_train[start:end]

        sess.run(train_step, feed_dict={
            xs: batch_xs,
            ys: batch_ys,
            keep_prob: keep_prob_rate
        })

        if i % 100 == 0:
            acc = compute_accuracy(x_test[:1000], y_test[:1000])
            print("step:", i, "accuracy:", acc)



step: 0 accuracy: 0.17
step: 100 accuracy: 0.879
step: 200 accuracy: 0.915
step: 300 accuracy: 0.94
step: 400 accuracy: 0.953
step: 500 accuracy: 0.955
step: 600 accuracy: 0.954
step: 700 accuracy: 0.958
step: 800 accuracy: 0.965
step: 900 accuracy: 0.964
step: 1000 accuracy: 0.965
step: 1100 accuracy: 0.966
step: 1200 accuracy: 0.965
step: 1300 accuracy: 0.967
step: 1400 accuracy: 0.968
step: 1500 accuracy: 0.966
step: 1600 accuracy: 0.969
step: 1700 accuracy: 0.971
step: 1800 accuracy: 0.967
step: 1900 accuracy: 0.973
